# Animation of Ice shelf colapse 

This notebook loads Landsat and Sentinel 2 data from the Microsoft Planetary Computer STAC catalogue and builds and exports an RGB animation.
This notebook requires a lot of memory and should be run using a large sandbox instance.

In [1]:
import sys
import pystac_client
import planetary_computer
import datacube
import odc.stac
import odc.geo.xr

import matplotlib.pyplot as plt
import numpy as np
import xarray as xr
import ipywidgets as widgets

from IPython.display import Image, display, clear_output
from IPython.core.display import Video
from odc.geo.geom import BoundingBox
from dea_tools.datahandling import load_ard
from dea_tools.plotting import display_map, rgb, xr_animation
from skimage.exposure import rescale_intensity, equalize_adapthist

sys.path.insert(1, "../../Tools/")

In [2]:
dc = datacube.Datacube(app="Planetary_computer")

# Find AOI

In [3]:
# Coordinates for Antactic Brunt Iceshelf

point_x, point_y = ( -75.5881, -26.5877)

lat = (point_x - 0.8, point_x + 0.3)
lon = (point_y - 5.0, point_y + 1)

# Convert data-cube style queries into something readable by `pystac_client`
bbox = BoundingBox.from_xy(lon, lat)

In [4]:
# Display area on map

display_map(x=lon, y=lat)

# Connect to Microsoft Planetary Computer (Sentinel 2 and Landsat)

In [5]:
# Open a client pointing to the Microsoft Planetary Computer data catalogue
catalog = pystac_client.Client.open(
    "https://planetarycomputer.microsoft.com/api/stac/v1",
    modifier=planetary_computer.sign_inplace,
)

Optional:

use a filter to limit scenes with poor data or a-lot of cloud cover. I found using this results in returning a lot of partial observations


In [ ]:
# Set up a filter query
# filter_query = { "op": "and",
#     "args":[ {"op": "lte", "args": [{"property": "eo:cloud_cover"}, 25]},
#             {"op": "lte", "args": [{"property": "s2:nodata_pixel_percentage"}, 50]}
#            ]}

# Find Landsat scenes

MPP collection `landsat-c2-l2` provides level 2 data for the combined Landsat sensors

All available images for the period 1 Dec 2022 to 31 March 2023 were manually reviewed.
Only clear images were loaded below.

In [6]:
ls_Dec1 = catalog.search(
    collections=["landsat-c2-l2"],
    datetime="2022-12-05/2022-12-08",
    intersects=bbox.boundary(),
    ).item_collection()

ls_Dec2 = catalog.search(
    collections=["landsat-c2-l2"],
    datetime="2022-12-11/2022-12-12",
    intersects=bbox.boundary(),
    ).item_collection()

ls_Dec3 = catalog.search(
    collections=["landsat-c2-l2"],
    datetime="2022-12-28/2022-12-29",
    intersects=bbox.boundary(),
    ).item_collection()

ls_Jan1 = catalog.search(
    collections=["landsat-c2-l2"],
    datetime="2023-01-08",
    intersects=bbox.boundary(),
    ).item_collection()

ls_Jan2 = catalog.search(
    collections=["landsat-c2-l2"],
    datetime="2023-01-10/2023-01-12",
    intersects=bbox.boundary(),
    ).item_collection()

ls_Jan3 = catalog.search(
    collections=["landsat-c2-l2"],
    datetime="2023-01-17",
    intersects=bbox.boundary(),
    ).item_collection()

ls_Jan4 = catalog.search(
    collections=["landsat-c2-l2"],
    datetime="2023-01-20/2023-01-21",
    intersects=bbox.boundary(),
    ).item_collection()

ls_Jan5 = catalog.search(
    collections=["landsat-c2-l2"],
    datetime="2023-01-24/2023-01-26",
    intersects=bbox.boundary(),
    ).item_collection()

ls_Feb1 = catalog.search(
    collections=["landsat-c2-l2"],
    datetime="2023-02-01/2023-02-05",
    intersects=bbox.boundary(),
    ).item_collection()

ls_Feb2 = catalog.search(
    collections=["landsat-c2-l2"],
    datetime="2023-02-07",
    intersects=bbox.boundary(),
    ).item_collection()

ls_Feb3 = catalog.search(
    collections=["landsat-c2-l2"],
    datetime="2023-02-09/2023-02-12",
    intersects=bbox.boundary(),
    ).item_collection()

ls_Feb4 = catalog.search(
    collections=["landsat-c2-l2"],
    datetime="2023-02-17/2023-02-18",
    intersects=bbox.boundary(),
    ).item_collection()

ls_Feb5 = catalog.search(
    collections=["landsat-c2-l2"],
    datetime="2023-02-20/2023-02-23",
    intersects=bbox.boundary(),
    ).item_collection()

ls_Feb6 = catalog.search(
    collections=["landsat-c2-l2"],
    datetime="2023-02-26/2023-02-27",
    intersects=bbox.boundary(),
    ).item_collection()



In [7]:
# Landsat - combine data
items_to_load_ls = (
    ls_Dec1 +
    ls_Dec2 +
    ls_Dec3 +
    ls_Jan1 +
    ls_Jan2 +
    ls_Jan3 +
    ls_Jan4 +
    ls_Jan5 +
    ls_Feb1 +
    ls_Feb2 +
    ls_Feb3 +
    ls_Feb4 +
    ls_Feb5 +
    ls_Feb6
    )

# Find Sentinel 2 scenes

MPP collection `sentinel-2-l2a` provides level 2 data for the combined S2A and S2B sensors (confirm whether or not C is included. Irrelevant for this data range anyway)

In [9]:
#before shelf collapse

items_neg_five = catalog.search(
    collections=["sentinel-2-l2a"],
    datetime="2022-12-05/2022-12-08",
    intersects=bbox.boundary(),
    ).item_collection()

items_neg_four = catalog.search(
    collections=["sentinel-2-l2a"],
    datetime="2022-12-11/2022-12-14",
    intersects=bbox.boundary(),
    ).item_collection()

items_neg_three = catalog.search(
    collections=["sentinel-2-l2a"],
    datetime="2022-12-28/2022-12-28",
    intersects=bbox.boundary(),
    ).item_collection()


items_neg_two = catalog.search(
    collections=["sentinel-2-l2a"],
    datetime="2023-01-20/2023-01-20",
    intersects=bbox.boundary(),
    ).item_collection()

items_neg_one = catalog.search(
    collections=["sentinel-2-l2a"],
    datetime="2023-01-10/2023-01-10",
    intersects=bbox.boundary(),
    ).item_collection()


#beginning of collapse is here:

items_one = catalog.search(
    collections=["sentinel-2-l2a"],
    datetime="2023-01-24/2023-01-25",
    intersects=bbox.boundary(),
    # filter=filter_query,
    ).item_collection()

items_two = catalog.search(
    collections=["sentinel-2-l2a"],
    datetime="2023-02-03/2023-02-05",
    intersects=bbox.boundary(),
    # filter=filter_query,
    ).item_collection()

items_three = catalog.search(
    collections=["sentinel-2-l2a"],
    datetime="2023-02-12/2023-02-12",
    intersects=bbox.boundary(),
    # filter=filter_query,
    ).item_collection()

items_four = catalog.search(
    collections=["sentinel-2-l2a"],
    datetime="2023-02-22/2023-02-23",
    intersects=bbox.boundary(),
    # filter=filter_query,
    ).item_collection()

items_five = catalog.search(
    collections=["sentinel-2-l2a"],
    datetime="2023-03-04/2023-03-05",
    intersects=bbox.boundary(),
    # filter=filter_query,
    ).item_collection()

In [10]:
# combine sentinel data into single dataset
items_to_load_s2 = items_neg_five + items_neg_four + items_neg_three + items_neg_two + items_neg_one + items_one + items_two + items_three + items_four + items_five

# Load Landsat scenes

In [1]:
# Load Landsat imagery with odc-stac

# Load imagery with odc-stac
ls_data = odc.stac.load(
    items_to_load,
    bbox=bbox,
    bands=["red","green", "blue"],
    crs="EPSG:3031",
    groupby="id",
    resolution=100) #re-sampling to 100 meters doesn't seem too bad for the animation

In [ ]:
# Make no-data 0 so it's plots black not white
ls_data = ls_data.where(ls_data > -1, 0)

In [ ]:
# Groupby day, taking max reflectance values to avoid scene boundary artefacts
ls_daily = ls_data.sortby('time').resample(time='1D').max()
ls_daily = ls_daily.dropna(dim='time', how='all')

In [ ]:
# check our loaded images IF neeeded
rgb(ls_daily, bands=["red","green", "blue"], col='time', col_wrap=4, titles=ls_daily.time)

# Load Sentinel 2 scenes

In [12]:
# Load Sentinel imagery with odc-stac
sent_data = odc.stac.load(
    items_to_load,
    bbox=bbox,
    bands=["red","green", "blue"],
    crs="EPSG:3031",
    groupby="id",
    resolution=100) #re-sampling to 100 meters doesn't seem too bad for the animation

In [ ]:
# Make no-data 0 so it's plots black not white
sent_data = sent_data.where(sent_data > -1, 0)

In [ ]:
# Groupby day, taking max reflectance values to avoid scene boundary artefacts
s2_daily = sent_data.sortby('time').resample(time='1D').max()
s2_daily = s2_daily.dropna(dim='time', how='all')


In [ ]:
# check our loaded images IF neeeded
rgb(s2_daily, bands=["red","green", "blue"], col='time', col_wrap=4, titles=s2_daily.time)

# Combine and cleanse data

In [ ]:
# combine datasets and sort by time 
recombined = xr.concat([ls_daily, s2_daily], dim='time').sortby('time')

In [73]:
# manually review imagery and identify scenes for removal by clicking `Mark for Deletion` button

# Get list of timesteps from YOUR dataset
timesteps = recombined.time.values
timesteps_to_delete = []

# Create interactive viewer
current_idx = [0]

# Create output widget for the plot
output = widgets.Output()

def plot_timestep(idx):
    with output:
        clear_output(wait=True)
        timestep = timesteps[idx]
        
        # Select this timestep
        data = recombined.isel(time=idx)
        
        # Stack the RGB bands - change band names to match yours
        rgb_array = xr.concat([data['red'], data['green'], data['blue']], dim='band')
        
        # Transpose so band is the last dimension (y, x, band)
        rgb_array = rgb_array.transpose(..., 'band')
        
        # Plot
        fig, ax = plt.subplots(figsize=(10, 8))
        rgb_array.plot.imshow(rgb='band', ax=ax, robust=True)
        ax.set_title(f"Timestep: {timestep} ({idx+1}/{len(timesteps)})")
        plt.show()
        
        print(f"\nCurrent timestep: {timestep} ({idx+1}/{len(timesteps)})")
        print(f"Timesteps marked for deletion: {len(timesteps_to_delete)}")

def next_timestep(b):
    if current_idx[0] < len(timesteps) - 1:
        current_idx[0] += 1
        plot_timestep(current_idx[0])

def prev_timestep(b):
    if current_idx[0] > 0:
        current_idx[0] -= 1
        plot_timestep(current_idx[0])

def mark_delete(b):
    timestep = timesteps[current_idx[0]]
    if timestep not in timesteps_to_delete:
        timesteps_to_delete.append(timestep)
        with output:
            print(f"Marked timestep {timestep} for deletion")

# Create buttons
next_btn = widgets.Button(description="Next")
prev_btn = widgets.Button(description="Previous")
delete_btn = widgets.Button(description="Mark for Deletion", button_style='danger')

next_btn.on_click(next_timestep)
prev_btn.on_click(prev_timestep)
delete_btn.on_click(mark_delete)

# Display buttons and output area
display(widgets.HBox([prev_btn, next_btn, delete_btn]))
display(output)

# Show first plot
plot_timestep(0)

Output()

In [74]:
# Remove marked timesteps
recombined_cleaned = recombined.sel(time=~recombined.time.isin(timesteps_to_delete))

# Export satellite data

Export your xarray stacks of pre and post cleansed imagery to enable future redevelopment of the animation

In [ ]:
# Save data to avoid re-loading in the future.
# If this cell fails, ensure that no other file by the same name exists in your current file directory.

# pre-cleansed data
recombined.to_zarr('Brunt_iceshelf_ls_s2_imagery.zarr')
# post-cleansed data
recombined_cleaned.to_zarr('Brunt_iceshelf_ls_s2_imagery_cleansed.zarr')

# Import satellite data

If you wish to revisit the imagery stack used to create the animation, re-load the data in xarray

In [ ]:
# # open data
# recombined = xr.open_zarr('Brunt_iceshelf_ls_s2_imagery.zarr')
# recombined_cleansed = xr.open_zarr('Brunt_iceshelf_ls_s2_imagery_cleaned.zarr')

# Animate

In [76]:
# Animate remaining, time-sorted, manually filtered imagery, adding contrast enhancement to support a clear animation

custom_funcs = [
    rescale_intensity, # Return image after stretching or shrinking its intensity levels.
    equalize_adapthist # An algorithm for local contrast enhancement
]

# Set this as your unique pathname. File will output to current folder
output_path = "Brunt_iceshelf_collapse_animated_timeseries_S2_LS_clipped.mp4"

xr_animation(
    ds=recombined_cleaned,
    bands=["red","green", "blue"],
    output_path=output_path,
    percentile_stretch=(0.05, 0.95),
    show_date= False,
    image_proc_funcs=custom_funcs,
    interval=600,
    width_pixels=1000,
)

# Plot animation
plt.close()
Video(output_path, embed=True)

Applying custom image processing functions


  0%|          | 0/19 (0.0 seconds remaining at ? frames/s)

Exporting animation to Brunt_iceshelf_collapse_animated_timeseries_S2_LS_clipped.mp4


  0%|          | 0/19 (0.0 seconds remaining at ? frames/s)